# ASR evaluation on collected ad videos

Downloads Instagram Reels from your CSV, transcribes them with faster-whisper,
and measures how much of the spoken content — and how much of the **persuasion
signal** — survives transcription.

Run cells in order. Stage 3 requires you to listen to the audio; there is no way
to automate it, because a machine transcript cannot be its own ground truth.

## Setup

In [1]:
!pip install yt-dlp faster-whisper matplotlib -q
print("installed")

installed


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Program Files\Python313\python.exe -m pip install --upgrade pip


In [2]:
import csv, json, re, subprocess, sys, time, unicodedata
from pathlib import Path

# ---- EDIT THIS ----
CSV_PATH = r"C:\Users\shash\Downloads\Video_dataset.csv"
# -------------------

ROOT  = Path("asr_study")
AUDIO = ROOT / "audio"
AUTO  = ROOT / "auto"
WORK  = ROOT / "work"
for d in (AUDIO, AUTO, WORK):
    d.mkdir(parents=True, exist_ok=True)

print("csv found" if Path(CSV_PATH).exists() else f"CSV NOT FOUND: {CSV_PATH}")
print(f"working folder: {ROOT.resolve()}")

csv found
working folder: C:\Users\shash\Downloads\asr_study


## Scoring helpers

Normalisation strips casing and punctuation, which ASR does not reliably produce.
Scoring those would inflate the error rate for reasons unrelated to lost words.

In [3]:
TRIGGER_PHRASES = [
    "act now", "limited time", "hurry", "don't miss", "last chance",
    "only", "left in stock", "selling fast", "running out", "while supplies last",
    "doctors", "experts", "clinically proven", "scientifically proven", "proven",
    "guaranteed", "miracle", "breakthrough", "secret", "we recommend",
    "thousands of customers", "best seller", "everyone", "risk free",
    "money back", "before it's too late", "transform",
]

_PUNCT = re.compile(r"[^\w\s']")
_SPACE = re.compile(r"\s+")

def normalize(t):
    t = unicodedata.normalize("NFKC", t).lower().replace("\u2019", "'")
    return _SPACE.sub(" ", _PUNCT.sub(" ", t)).strip()

def tokenize(t):
    return normalize(t).split()

def align(ref, hyp):
    n, m = len(ref), len(hyp)
    d = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): d[i][0] = i
    for j in range(m+1): d[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            d[i][j] = (d[i-1][j-1] if ref[i-1]==hyp[j-1]
                       else 1 + min(d[i-1][j-1], d[i-1][j], d[i][j-1]))
    ops, i, j = [], n, m
    while i > 0 or j > 0:
        if i>0 and j>0 and ref[i-1]==hyp[j-1] and d[i][j]==d[i-1][j-1]:
            ops.append(("ok", ref[i-1], hyp[j-1])); i, j = i-1, j-1
        elif i>0 and j>0 and d[i][j]==d[i-1][j-1]+1:
            ops.append(("sub", ref[i-1], hyp[j-1])); i, j = i-1, j-1
        elif i>0 and d[i][j]==d[i-1][j]+1:
            ops.append(("del", ref[i-1], None)); i -= 1
        else:
            ops.append(("ins", None, hyp[j-1])); j -= 1
    ops.reverse()
    return (sum(1 for t,_,_ in ops if t=="sub"),
            sum(1 for t,_,_ in ops if t=="del"),
            sum(1 for t,_,_ in ops if t=="ins"), ops)

def trigger_survival(ref_text, hyp_text):
    r, h = normalize(ref_text), normalize(hyp_text)
    present  = [p for p in TRIGGER_PHRASES if normalize(p) in r]
    survived = [p for p in present if normalize(p) in h]
    return present, survived, [p for p in present if p not in survived]

def read_ref(path):
    text = Path(path).read_text(encoding="utf-8")
    m = re.search(r"^REF:\s*(.*)$", text, flags=re.MULTILINE|re.DOTALL)
    if not m: return None
    ref = re.sub(r"\[inaudible\]", "", m.group(1).strip(), flags=re.I)
    return _SPACE.sub(" ", ref).strip()

print("helpers loaded")

helpers loaded


## Stage 1 — download

Reads your CSV, finds the URL column automatically, and pulls 16kHz mono audio
with yt-dlp.

Expect failures. Instagram rate-limits hard and some Reels will be private or
removed. Re-run this cell to retry — the manifest prevents duplicate work.
Raise `SLEEP` if a lot of them fail.

In [4]:
SLEEP = 5   # seconds between downloads; raise to 10-15 if you get throttled

def find_url_column(header, rows):
    for i, name in enumerate(header):
        if any(k in name.lower() for k in ("url","link","reel","video")):
            if any(r[i].strip().startswith("http") for r in rows[:10] if len(r)>i):
                return i
    for i in range(len(header)):
        if any(r[i].strip().startswith("http") for r in rows[:10] if len(r)>i):
            return i
    return None

with open(CSV_PATH, newline="", encoding="utf-8-sig") as f:
    rows = list(csv.reader(f))

if rows[0] and rows[0][0].strip().startswith("http"):
    header, data, idx = ["url"], rows, 0      # no header row
    print("No header row detected — treating every row as data.")
else:
    header, data = rows[0], rows[1:]
    idx = find_url_column(header, data)
if idx is None:
    raise SystemExit(f"No URL column found. Header: {header}")

urls = [r[idx].strip() for r in data if len(r)>idx and r[idx].strip().startswith("http")]
print(f"{len(urls)} URLs found in column '{header[idx] if idx<len(header) else idx}'\n")

mf = ROOT / "manifest.json"
manifest = json.loads(mf.read_text()) if mf.exists() else {}
ok = fail = skip = 0

for n, url in enumerate(urls, 1):
    stem = f"ad{n:02d}"
    if list(AUDIO.glob(f"{stem}.*")):
        skip += 1; continue
    print(f"[{n}/{len(urls)}] {stem}  {url[:55]}")
    cmd = ["yt-dlp", url,
           "-f", "bestaudio/best",
           "-o", str(AUDIO / f"{stem}.%(ext)s"),
           "--no-playlist", "--quiet", "--no-warnings",
           "--sleep-interval", str(SLEEP), "--max-sleep-interval", str(SLEEP*2)]
    try:
        subprocess.run(cmd, check=True, timeout=180)
        manifest[stem] = url; ok += 1; print("      ok")
    except subprocess.CalledProcessError:
        fail += 1; print("      FAILED (private, removed, or rate-limited)")
    except subprocess.TimeoutExpired:
        fail += 1; print("      TIMEOUT")
    except FileNotFoundError:
        raise SystemExit("yt-dlp not installed")

mf.write_text(json.dumps(manifest, indent=2))
print(f"\n{ok} downloaded, {skip} already present, {fail} failed")

No header row detected — treating every row as data.
42 URLs found in column 'url'

[8/42] ad08  https://www.instagram.com/reel/DXCGJz_jQ4w/?igsh=MWdnOT
      ok
[9/42] ad09  https://www.instagram.com/reel/DbAzgopCZdm/?igsh=bWgxNW
      ok
[10/42] ad10  https://www.instagram.com/p/DZ3o2KgMsB-/?igsh=Ym96eTcwe
      ok
[11/42] ad11  https://www.instagram.com/reel/DaDjgBlt0ha/?igsh=MWQwZm
      ok
[12/42] ad12  https://www.instagram.com/reel/DbBfNEryb43/
      ok
[13/42] ad13  https://www.instagram.com/reel/DbBfNEryb43/
      ok
[14/42] ad14  https://www.instagram.com/reel/DangpY9vrXn/
      ok
[15/42] ad15  https://www.instagram.com/reel/DangpY9vrXn/
      ok
[16/42] ad16  https://www.instagram.com/reel/Da66bdzCmgC/
      ok
[17/42] ad17  https://www.instagram.com/reel/DasnSlDzJrE/?igsh=MTNxdW
      ok
[18/42] ad18  https://www.instagram.com/p/Dbqie8GAorl/?igsh=dTZ3MzRta
      ok
[19/42] ad19  https://www.instagram.com/reel/DaCD0ogMoz_/?igsh=bGRkdm
      ok
[20/42] ad20  https://www.inst

## Stage 2 — transcribe

This is the output being **tested**, not the answer. Start with `base`; try
`small` later if you want a model-size comparison for the report.

In [5]:
from faster_whisper import WhisperModel

MODEL_SIZE = "base"

AUDIO_EXTS = {".webm", ".m4a", ".mp4", ".opus", ".wav", ".mp3", ".aac", ".ogg"}
files = sorted(p for p in AUDIO.iterdir() if p.suffix.lower() in AUDIO_EXTS)
if not files:
    raise SystemExit("No audio downloaded yet — run stage 1")

print(f"Loading '{MODEL_SIZE}' ...")
model = WhisperModel(MODEL_SIZE, device="cpu", compute_type="int8")

for n, f in enumerate(files, 1):
    t0 = time.time()
    segs, info = model.transcribe(str(f))
    text = " ".join(s.text for s in segs).strip()
    (AUTO / f"{f.stem}.txt").write_text(text, encoding="utf-8")
    print(f"[{n}/{len(files)}] {f.stem}  {len(text.split()):4d} words  "
          f"{time.time()-t0:5.1f}s  lang={info.language}")

print(f"\nTranscripts in {AUTO}")

c:\Users\shash\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading 'base' ...
[1/41] ad01    33 words    3.7s  lang=en
[2/41] ad02   593 words   53.9s  lang=en
[3/41] ad03    46 words    4.0s  lang=en
[4/41] ad04   139 words    8.0s  lang=en
[5/41] ad05    16 words    3.3s  lang=en
[6/41] ad06   131 words    9.5s  lang=en
[7/41] ad07   113 words    8.0s  lang=en
[8/41] ad08   113 words    7.1s  lang=en
[9/41] ad09   108 words    6.9s  lang=en
[10/41] ad10   106 words    7.4s  lang=en
[11/41] ad11    71 words    5.9s  lang=en
[12/41] ad12   113 words    8.1s  lang=en
[13/41] ad13   113 words    7.8s  lang=en
[14/41] ad14   113 words    8.3s  lang=en
[15/41] ad15   113 words    7.6s  lang=en
[16/41] ad16    82 words    4.6s  lang=en
[17/41] ad17   186 words   11.6s  lang=en
[18/41] ad18   113 words    9.2s  lang=en
[19/41] ad19   137 words    9.2s  lang=en
[20/41] ad20    48 words    3.8s  lang=en
[21/41] ad21   142 words    8.0s  lang=en
[22/41] ad22    53 words    3.9s  lang=en
[23/41] ad23    46 words    3.6s  lang=en
[24/41] ad24   118 words

## Stage 3 — build correction worksheets

Creates one `.md` per clip in `asr_study/work/`, pre-filled with the whisper
draft so you edit rather than type from scratch.

In [6]:
HEADER = """# Reference — {stem}

Source: {url}
Transcribed by: [your name]
Date: [date]

The REF line below came from faster-whisper and is NOT ground truth.
Play the matching file in asr_study/audio/ and correct it by ear.

- Exactly what is said, including filler and false starts.
- Numbers as spoken: "twenty percent", not "20%".
- Mark unclear audio as [inaudible].
- Listen hardest at the START and END — clipped words are the failure being
  measured, and they leave no trace in the draft to alert you.

---

REF: {draft}
"""

mf = ROOT / "manifest.json"
manifest = json.loads(mf.read_text()) if mf.exists() else {}
autos = sorted(AUTO.glob("*.txt"))
if not autos:
    raise SystemExit("No transcripts — run stage 2")

made = 0
for f in autos:
    target = WORK / f"{f.stem}.ref.md"
    if target.exists():
        continue
    target.write_text(HEADER.format(stem=f.stem, url=manifest.get(f.stem, "[url]"),
                                    draft=f.read_text(encoding="utf-8").strip()),
                      encoding="utf-8")
    made += 1

words = sum(len(f.read_text(encoding="utf-8").split()) for f in autos)
print(f"{made} new worksheets ({len(autos)} clips, ~{words} draft words)")
print(f"Estimated correction time: {words/60:.0f}-{words/30:.0f} minutes")
print(f"\nOpen: {WORK.resolve()}")
print("\nDo 10 first, then run stage 4. Ten is enough for a defensible number.")

41 new worksheets (41 clips, ~5278 draft words)
Estimated correction time: 88-176 minutes

Open: C:\Users\shash\Downloads\asr_study\work

Do 10 first, then run stage 4. Ten is enough for a defensible number.


## Correct the worksheets now

Open `asr_study/work/` in VS Code or Notepad and fix each `REF:` line while
listening to the matching `.wav`.

**Bias warning.** Correcting a whisper draft means errors that *sound plausible*
can slip past unnoticed. Since faster-whisper is the same model family, a shared
mistake would never register as an error and would inflate your accuracy figure.
Listen properly rather than skimming the text.

Run the cell below to check progress.

In [7]:
done, todo = [], []
for s in sorted(WORK.glob("*.ref.md")):
    stem = s.name.replace(".ref.md", "")
    auto = AUTO / f"{stem}.txt"
    if not auto.exists(): continue
    ref = read_ref(s)
    if ref and normalize(ref) != normalize(auto.read_text(encoding="utf-8").strip()):
        done.append(stem)
    else:
        todo.append(stem)

print(f"corrected: {len(done)}   still identical to draft: {len(todo)}")
if todo: print("\ntodo:", ", ".join(todo[:15]) + (" ..." if len(todo)>15 else ""))

corrected: 0   still identical to draft: 41

todo: ad01, ad02, ad03, ad04, ad05, ad06, ad07, ad08, ad09, ad10, ad11, ad12, ad13, ad14, ad15 ...


## Stage 4 — score

Worksheets identical to their draft are skipped, not scored — otherwise you
would be measuring whisper against itself.

In [10]:
import webbrowser, os
for f in sorted(WORK.glob("*.ref.md"))[:10]:
    print(f.resolve())

C:\Users\shash\Downloads\asr_study\work\ad01.ref.md
C:\Users\shash\Downloads\asr_study\work\ad02.ref.md
C:\Users\shash\Downloads\asr_study\work\ad03.ref.md
C:\Users\shash\Downloads\asr_study\work\ad04.ref.md
C:\Users\shash\Downloads\asr_study\work\ad05.ref.md
C:\Users\shash\Downloads\asr_study\work\ad06.ref.md
C:\Users\shash\Downloads\asr_study\work\ad07.ref.md
C:\Users\shash\Downloads\asr_study\work\ad08.ref.md
C:\Users\shash\Downloads\asr_study\work\ad09.ref.md
C:\Users\shash\Downloads\asr_study\work\ad10.ref.md


In [13]:
pip install ipywidgets

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 13.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 30.7 MB/s  0:00:00

   ---------------------------------------- 0/3 [widgetsnbextension]
   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -----------------


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# In-notebook correction tool — play audio, edit text, save, next.
# Requires: pip install ipywidgets

import ipywidgets as widgets
from IPython.display import Audio, display, clear_output

AUDIO_EXTS = {".webm", ".m4a", ".mp4", ".opus", ".wav", ".mp3", ".aac", ".ogg"}

def audio_for(stem):
    for p in AUDIO.iterdir():
        if p.stem == stem and p.suffix.lower() in AUDIO_EXTS:
            return p
    return None

def worksheet_path(stem):
    return WORK / f"{stem}.ref.md"

def save_ref(stem, new_ref):
    p = worksheet_path(stem)
    text = p.read_text(encoding="utf-8")
    new_ref = " ".join(new_ref.split())
    text = re.sub(r"^REF:.*$", "REF: " + new_ref.replace("\\", "\\\\"),
                  text, count=1, flags=re.MULTILINE | re.DOTALL)
    p.write_text(text, encoding="utf-8")

# Build the queue: only clips with real speech, uncorrected, shortest first.
queue = []
for s in sorted(WORK.glob("*.ref.md")):
    stem = s.name.replace(".ref.md", "")
    auto = AUTO / f"{stem}.txt"
    if not auto.exists():
        continue
    draft = auto.read_text(encoding="utf-8").strip()
    ref = read_ref(s) or ""
    if len(draft.split()) < 5:          # music-only or silent clip
        continue
    if normalize(ref) != normalize(draft):
        continue                         # already corrected
    queue.append((stem, len(draft.split())))

queue.sort(key=lambda x: x[1])           # shortest first — quick wins
queue = [q[0] for q in queue]

state = {"i": 0}

out = widgets.Output()
box = widgets.Textarea(layout=widgets.Layout(width="100%", height="160px"))
status = widgets.HTML()
save_btn = widgets.Button(description="Save + next", button_style="success")
skip_btn = widgets.Button(description="Skip")
back_btn = widgets.Button(description="Back")

def render():
    with out:
        clear_output(wait=True)
        if state["i"] >= len(queue):
            print("Queue complete. Re-run the scoring cell.")
            return
        stem = queue[state["i"]]
        a = audio_for(stem)
        draft = (AUTO / f"{stem}.txt").read_text(encoding="utf-8").strip()
        print(f"[{state['i']+1}/{len(queue)}]  {stem}   ({len(draft.split())} draft words)")
        print("Listen to the START and END especially — dropped words leave no trace.")
        if a:
            display(Audio(str(a)))
        else:
            print("(no audio file found)")
        box.value = draft

def on_save(_):
    if state["i"] < len(queue):
        save_ref(queue[state["i"]], box.value)
        status.value = f"<b>saved {queue[state['i']]}</b>"
        state["i"] += 1
        render()

def on_skip(_):
    state["i"] += 1
    render()

def on_back(_):
    state["i"] = max(0, state["i"] - 1)
    render()

save_btn.on_click(on_save); skip_btn.on_click(on_skip); back_btn.on_click(on_back)

print(f"{len(queue)} clips need correction (clips under 5 words skipped as music-only).")
print("Shortest first. Do ten, then score.\n")
display(out, box, widgets.HBox([save_btn, skip_btn, back_btn]), status)
render()

41 clips need correction (clips under 5 words skipped as music-only).
Shortest first. Do ten, then score.



Output()

Textarea(value='', layout=Layout(height='160px', width='100%'))

HTML(value='')

In [11]:
results, uncorrected = [], []

for s in sorted(WORK.glob("*.ref.md")):
    stem = s.name.replace(".ref.md", "")
    auto = AUTO / f"{stem}.txt"
    if not auto.exists(): continue
    ref_text = read_ref(s)
    hyp_text = auto.read_text(encoding="utf-8").strip()
    if not ref_text: continue
    if normalize(ref_text) == normalize(hyp_text):
        uncorrected.append(stem); continue

    ref, hyp = tokenize(ref_text), tokenize(hyp_text)
    if not ref: continue
    subs, dels, ins, ops = align(ref, hyp)
    err = subs + dels + ins
    present, survived, lost = trigger_survival(ref_text, hyp_text)
    results.append(dict(file=stem, ref_words=len(ref), hyp_words=len(hyp),
                        sub=subs, dele=dels, ins=ins, wer=err/len(ref),
                        accuracy=max(0,1-err/len(ref)),
                        triggers_present=len(present),
                        triggers_survived=len(survived),
                        triggers_lost="; ".join(lost), ops=ops))

if uncorrected:
    print(f"Skipped {len(uncorrected)} uncorrected: {', '.join(uncorrected[:8])}"
          f"{' ...' if len(uncorrected)>8 else ''}\n")
if not results:
    raise SystemExit("Nothing scored — correct at least one worksheet by ear first.")

print(f"{'file':<10}{'ref':>6}{'sub':>6}{'del':>6}{'ins':>6}{'WER':>9}{'acc':>9}{'trig':>9}")
print("-"*61)
for r in results:
    trig = f"{r['triggers_survived']}/{r['triggers_present']}" if r['triggers_present'] else "-"
    print(f"{r['file']:<10}{r['ref_words']:>6}{r['sub']:>6}{r['dele']:>6}"
          f"{r['ins']:>6}{r['wer']:>8.1%}{r['accuracy']:>9.1%}{trig:>9}")

TW = sum(r["ref_words"] for r in results)
TS = sum(r["sub"] for r in results); TD = sum(r["dele"] for r in results)
TI = sum(r["ins"] for r in results); TE = TS+TD+TI
TP = sum(r["triggers_present"] for r in results)
TSV= sum(r["triggers_survived"] for r in results)
WER = TE/TW if TW else 0

print("-"*61)
print(f"{'OVERALL':<10}{TW:>6}{TS:>6}{TD:>6}{TI:>6}{WER:>8.1%}{1-WER:>9.1%}"
      f"{f'{TSV}/{TP}' if TP else '-':>9}")
print(f"\nn = {len(results)} ads, {TW} reference words")
if TE: print(f"Deletions: {TD/TE:.0%} of all errors")
if TP: print(f"Trigger phrases retained: {TSV}/{TP} ({TSV/TP:.0%})")

Skipped 41 uncorrected: ad01, ad02, ad03, ad04, ad05, ad06, ad07, ad08 ...



SystemExit: Nothing scored — correct at least one worksheet by ear first.

In [ ]:
print("What these numbers support\n" + "="*55)
if WER < 0.10 and TP and TSV/TP > 0.9:
    print("They do NOT support dropping audio on quality grounds.\n"
          "Transcription held up. If the modality was cut, the honest\n"
          "reason is scope and effort — write that instead.")
elif TD and TE and TD/TE > 0.5:
    print("Deletions dominate, consistent with words clipped at capture\n"
          "boundaries rather than misheard. Supports the week 5 blocker\n"
          "as originally described.")
else:
    print("Errors are mixed rather than deletion-dominated. Points at\n"
          "general recognition difficulty (accents, music, technical terms)\n"
          "rather than chunk-boundary truncation. Describe what you found.")

## Word-level differences

In [ ]:
for r in results[:3]:
    print(f"\n{r['file']} — WER {r['wer']:.1%}\n" + "-"*45)
    shown = 0
    for tag, ref_w, hyp_w in r["ops"]:
        if tag == "ok": continue
        if tag == "sub":  print(f"  heard '{hyp_w}' instead of '{ref_w}'")
        elif tag == "del": print(f"  DROPPED '{ref_w}'")
        else:              print(f"  inserted '{hyp_w}'")
        shown += 1
        if shown >= 25:
            print("  ..."); break
    if shown == 0: print("  exact match")

## Chart + export

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
correct = TW - TS - TD
ax1.bar(["correct","substituted","deleted","inserted"], [correct, TS, TD, TI],
        color=["#5DCAA5","#EF9F27","#E24B4A","#85B7EB"])
ax1.set_ylabel("words"); ax1.set_title(f"Transcription outcome — WER {WER:.1%}")

if TP:
    ax2.bar(["survived","lost"], [TSV, TP-TSV], color=["#5DCAA5","#E24B4A"])
    ax2.set_ylabel("trigger phrases")
    ax2.set_title(f"Persuasion signal — {TSV}/{TP} retained")
else:
    ax2.text(.5,.5,"no trigger phrases", ha="center", va="center"); ax2.axis("off")

plt.tight_layout(); plt.savefig("asr_eval.png", dpi=150); plt.show()

with open("results.csv","w",newline="",encoding="utf-8") as f:
    fields = [k for k in results[0] if k != "ops"]
    w = csv.DictWriter(f, fieldnames=fields); w.writeheader()
    for r in results: w.writerow({k:r[k] for k in fields})
print("saved asr_eval.png and results.csv")

## Paragraph for the weekly report

In [ ]:
lost_all = [l for r in results for l in r["triggers_lost"].split("; ") if l]
lost_str = ", ".join(f'"{p}"' for p in sorted(set(lost_all))[:6]) if lost_all else "none"
MODEL_SIZE = globals().get("MODEL_SIZE", "base")

print(f"""Audio transcription — measured

We hand-transcribed {len(results)} video ads ({TW} reference words) and compared
them against the faster-whisper capture pipeline (model: {MODEL_SIZE}).

Word accuracy was {1-WER:.1%} (WER {WER:.1%}). Deletions accounted for
{TD/TE:.0%} of all errors.

Of {TP} persuasion trigger phrases present in the reference transcripts,
{TSV} survived transcription ({TSV/TP:.0%}). Phrases lost included: {lost_str}.

Because both the classifier and the trigger-phrase matcher depend on this
vocabulary, overall word accuracy overstates how usable the audio path is.""")